# 01 · Data Ingestion
Read the raw NASA C-MAPSS text files, assign the schema, and persist to Parquet.

**Prerequisite:** run `bash scripts/download_data.sh` (or `scripts\download_data.ps1`) so the files are in `data/raw/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # project root, so `import src` works

In [2]:
from src.spark_utils import get_spark
from src.ingest import ingest
from src.config import SUBSETS
spark = get_spark()

Ingest one subset and inspect the typed schema:

In [3]:
df = ingest(spark, 'FD001')
df.printSchema()
df.show(5)

root
 |-- unit_nr: long (nullable = true)
 |-- time_cycles: long (nullable = true)
 |-- setting_1: double (nullable = true)
 |-- setting_2: double (nullable = true)
 |-- setting_3: double (nullable = true)
 |-- s_1: double (nullable = true)
 |-- s_2: double (nullable = true)
 |-- s_3: double (nullable = true)
 |-- s_4: double (nullable = true)
 |-- s_5: double (nullable = true)
 |-- s_6: double (nullable = true)
 |-- s_7: double (nullable = true)
 |-- s_8: double (nullable = true)
 |-- s_9: double (nullable = true)
 |-- s_10: double (nullable = true)
 |-- s_11: double (nullable = true)
 |-- s_12: double (nullable = true)
 |-- s_13: double (nullable = true)
 |-- s_14: double (nullable = true)
 |-- s_15: double (nullable = true)
 |-- s_16: double (nullable = true)
 |-- s_17: double (nullable = true)
 |-- s_18: double (nullable = true)
 |-- s_19: double (nullable = true)
 |-- s_20: double (nullable = true)
 |-- s_21: double (nullable = true)

+-------+-----------+---------+---------+-----

Ingest all four subsets and report their size (writes `data/processed/parquet/`):

In [4]:
total_e = total_r = 0
for s in SUBSETS:
    d = ingest(spark, s)
    e, r = d.select('unit_nr').distinct().count(), d.count()
    print(f'{s}: {e} engines, {r} rows')
    total_e += e; total_r += r
print(f'TOTAL: {total_e} engines, {total_r} rows')

FD001: 100 engines, 20631 rows
FD002: 260 engines, 53759 rows
FD003: 100 engines, 24720 rows
FD004: 249 engines, 61249 rows
TOTAL: 709 engines, 160359 rows


Expected: **709 engines, 160,359 rows** across the four training subsets.

In [5]:
spark.stop()